# Notebook 23 — Submission Package Builder and Validation

## The Pokémon Company — PTCG AI Battle Challenge

### Team Jesus

Notebook 22 validated the complete production agent.

Notebook 23 builds a clean competition submission package, verifies every required file,
checks Python imports, and creates a reusable submission archive.

## Objectives

1. Locate all production exports.
2. Define the submission directory structure.
3. Copy only required runtime files.
4. Include the validated 60-card deck.
5. Create the final agent entry point.
6. Validate Python syntax.
7. Validate imports in an isolated subprocess.
8. Test deck-request behavior.
9. Test action-selection behavior.
10. Detect missing or oversized files.
11. Create a submission manifest.
12. Build a ZIP archive.
13. Verify archive contents.
14. Produce a completion report.

# Cell 2 — Imports

In [1]:
from __future__ import annotations

import hashlib
import json
import shutil
import subprocess
import sys
import zipfile

from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Any

print("Python:", sys.version)
print("Working directory:", Path.cwd())

Python: 3.13.3 (tags/v3.13.3:6280bb5, Apr  8 2025, 14:47:33) [MSC v.1943 64 bit (AMD64)]
Working directory: D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge\notebooks


# Cell 3 — Locate the project

In [2]:
def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()

    markers = {
        "notebooks",
        "scripts",
        "src",
        "data",
    }

    for candidate in [current, *current.parents]:
        found = {
            marker
            for marker in markers
            if (candidate / marker).exists()
        }

        if len(found) >= 3:
            return candidate

    return current


PROJECT_ROOT = find_project_root()

SUBMISSION_ROOT = (
    PROJECT_ROOT
    / "submission_work"
    / "team_jesus_notebook23"
)

PACKAGE_DIR = SUBMISSION_ROOT / "package"

REPORT_DIR = (
    PROJECT_ROOT
    / "reports"
    / "notebook23"
)

ARCHIVE_FILE = (
    PROJECT_ROOT
    / "submission_work"
    / "team_jesus_notebook23.zip"
)

for directory in [
    SUBMISSION_ROOT,
    PACKAGE_DIR,
    REPORT_DIR,
]:
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )

print("Project root:", PROJECT_ROOT)
print("Submission root:", SUBMISSION_ROOT)
print("Package directory:", PACKAGE_DIR)
print("Archive:", ARCHIVE_FILE)
print("Reports:", REPORT_DIR)

Project root: D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge
Submission root: D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge\submission_work\team_jesus_notebook23
Package directory: D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge\submission_work\team_jesus_notebook23\package
Archive: D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge\submission_work\team_jesus_notebook23.zip
Reports: D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge\reports\notebook23


## Cell 4 — Define required source files

In [3]:
SOURCE_FILES = {
    "agent_export": (
        PROJECT_ROOT
        / "src"
        / "kaggle_agent"
        / "notebook21_export.py"
    ),
    "evaluation_export": (
        PROJECT_ROOT
        / "src"
        / "evaluation_harness"
        / "notebook22_export.py"
    ),
    "policy_export": (
        PROJECT_ROOT
        / "src"
        / "policy_engine"
        / "notebook20_export.py"
    ),
    "deck": (
        PROJECT_ROOT
        / "data"
        / "raw"
        / "kaggle_sample_submission"
        / "deck.csv"
    ),
}

for name, path in SOURCE_FILES.items():
    print(
        f"{'[FOUND]' if path.is_file() else '[MISSING]'} "
        f"{name}: {path}"
    )

missing_source_files = [
    str(path)
    for path in SOURCE_FILES.values()
    if not path.is_file()
]

if missing_source_files:
    raise FileNotFoundError(
        "Required submission sources are missing:\n"
        + "\n".join(missing_source_files)
    )

print("\nAll submission source files located.")

[FOUND] agent_export: D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge\src\kaggle_agent\notebook21_export.py
[FOUND] evaluation_export: D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge\src\evaluation_harness\notebook22_export.py
[FOUND] policy_export: D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge\src\policy_engine\notebook20_export.py
[FOUND] deck: D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge\data\raw\kaggle_sample_submission\deck.csv

All submission source files located.


## Cell 5 — Inspect source sizes

In [4]:
source_sizes = {
    name: path.stat().st_size
    for name, path in SOURCE_FILES.items()
}

for name, size in source_sizes.items():
    print(f"{name:20} {size:>10,} bytes")

total_source_size = sum(source_sizes.values())

print()
print("Total source size:", f"{total_source_size:,}", "bytes")

assert total_source_size > 0

agent_export             15,235 bytes
evaluation_export        13,970 bytes
policy_export            16,163 bytes
deck                        245 bytes

Total source size: 45,613 bytes


## Cell 6 — Create the submission package layout

In [5]:
from pathlib import Path
import shutil

PACKAGE_STRUCTURE = {
    "agent": PACKAGE_DIR / "agent",
    "evaluation": PACKAGE_DIR / "evaluation",
    "policy": PACKAGE_DIR / "policy",
    "data": PACKAGE_DIR / "data",
}

for folder in PACKAGE_STRUCTURE.values():
    folder.mkdir(
        parents=True,
        exist_ok=True,
    )

print("Package folders:")

for name, folder in PACKAGE_STRUCTURE.items():
    print(f"  {name:12} -> {folder}")

assert all(folder.exists() for folder in PACKAGE_STRUCTURE.values())

print("\nSubmission folder structure created.")

Package folders:
  agent        -> D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge\submission_work\team_jesus_notebook23\package\agent
  evaluation   -> D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge\submission_work\team_jesus_notebook23\package\evaluation
  policy       -> D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge\submission_work\team_jesus_notebook23\package\policy
  data         -> D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge\submission_work\team_jesus_notebook23\package\data

Submission folder structure created.


## Cell 7 — Copy the production files

In [6]:
COPIED_FILES = {}

copy_targets = {
    "agent_export": PACKAGE_STRUCTURE["agent"] / "notebook21_export.py",
    "evaluation_export": PACKAGE_STRUCTURE["evaluation"] / "notebook22_export.py",
    "policy_export": PACKAGE_STRUCTURE["policy"] / "notebook20_export.py",
    "deck": PACKAGE_STRUCTURE["data"] / "deck.csv",
}

for name, destination in copy_targets.items():

    shutil.copy2(
        SOURCE_FILES[name],
        destination,
    )

    COPIED_FILES[name] = destination

    print(f"[COPIED] {destination}")

assert len(COPIED_FILES) == len(copy_targets)

print("\nProduction files copied successfully.")

[COPIED] D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge\submission_work\team_jesus_notebook23\package\agent\notebook21_export.py
[COPIED] D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge\submission_work\team_jesus_notebook23\package\evaluation\notebook22_export.py
[COPIED] D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge\submission_work\team_jesus_notebook23\package\policy\notebook20_export.py
[COPIED] D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge\submission_work\team_jesus_notebook23\package\data\deck.csv

Production files copied successfully.


## Cell 8 — Verify copied files

In [7]:
for name, path in COPIED_FILES.items():

    print(
        f"{'[OK]' if path.exists() else '[MISSING]'}",
        path.name,
        path.stat().st_size,
        "bytes",
    )

assert all(path.exists() for path in COPIED_FILES.values())

print("\nCopied files verified.")

[OK] notebook21_export.py 15235 bytes
[OK] notebook22_export.py 13970 bytes
[OK] notebook20_export.py 16163 bytes
[OK] deck.csv 245 bytes

Copied files verified.


## Cell 9 — Compute SHA256 checksums

In [8]:
checksums = {}

for name, path in COPIED_FILES.items():

    digest = hashlib.sha256(
        path.read_bytes()
    ).hexdigest()

    checksums[name] = digest

    print(name)
    print(digest)
    print()

assert len(checksums) == 4

print("Checksums generated.")

agent_export
455a6fc88760cbe638e48806673dde179007c6bc0242d66f02a6c415e05a1709

evaluation_export
b8eee4d428b38710930f52e5f242441e1491c101e366fc66e2e9096398170795

policy_export
cc96a7448df796f56c9f0b6d8ab6b5dd1f54ae18fcd47aadd3a636506d4b490c

deck
b4464eb525a25e6598a972d00efc5e5b5156372e77f51853f4076d8ebb34fd7d

Checksums generated.


## Cell 10 — Create the package manifest

In [11]:
# Cell 10 — Create and validate the package manifest

deck_path = COPIED_FILES["deck"]

deck_ids = tuple(
    int(line.strip())
    for line in deck_path.read_text(
        encoding="utf-8-sig"
    ).splitlines()
    if line.strip()
)

manifest = {
    "project": "PTCG AI Battle Challenge",
    "team": "Team Jesus",
    "repository_cards": 1267,
    "official_cards": 1267,
    "deck_size": len(deck_ids),
    "files": {
        name: {
            "filename": path.name,
            "relative_path": str(
                path.relative_to(PACKAGE_DIR)
            ).replace("\\", "/"),
            "sha256": checksums[name],
            "size": path.stat().st_size,
        }
        for name, path in COPIED_FILES.items()
    },
}

print("Manifest created.")
print()
print(
    "Repository cards:",
    manifest["repository_cards"],
)
print(
    "Official cards:",
    manifest["official_cards"],
)
print("Deck size:", manifest["deck_size"])
print()
print("Files:")

for name, details in manifest["files"].items():
    print(
        f" - {name}: "
        f"{details['relative_path']} "
        f"({details['size']:,} bytes)"
    )

assert manifest["repository_cards"] == 1267
assert manifest["official_cards"] == 1267
assert manifest["deck_size"] == 60
assert len(manifest["files"]) == 4
assert len(set(deck_ids)) > 1

print("\nManifest validation passed.")

Manifest created.

Repository cards: 1267
Official cards: 1267
Deck size: 60

Files:
 - agent_export: agent/notebook21_export.py (15,235 bytes)
 - evaluation_export: evaluation/notebook22_export.py (13,970 bytes)
 - policy_export: policy/notebook20_export.py (16,163 bytes)
 - deck: data/deck.csv (245 bytes)

Manifest validation passed.


# Cell 11

In [13]:
import json

MANIFEST_FILE = PACKAGE_DIR / "manifest.json"

MANIFEST_FILE.write_text(
    json.dumps(
        manifest,
        indent=4,
    ),
    encoding="utf-8",
)

print(MANIFEST_FILE)

assert MANIFEST_FILE.exists()

print("\nManifest written successfully.")

D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge\submission_work\team_jesus_notebook23\package\manifest.json

Manifest written successfully.


# Cell 12 - Zip file

In [14]:
ZIP_FILE = (
    PROJECT_ROOT
    / "submission_work"
    / "team_jesus_submission.zip"
)

if ZIP_FILE.exists():
    ZIP_FILE.unlink()

shutil.make_archive(
    str(ZIP_FILE.with_suffix("")),
    "zip",
    PACKAGE_DIR,
)

print(ZIP_FILE)

assert ZIP_FILE.exists()

print("\nSubmission archive created.")

D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge\submission_work\team_jesus_submission.zip

Submission archive created.


# Cell 13

In [15]:
import zipfile

with zipfile.ZipFile(ZIP_FILE) as archive:
    archive_names = archive.namelist()

print("Archive contents:\n")

for name in archive_names:
    print(name)

assert len(archive_names) >= 5

print("\nArchive verification passed.")

Archive contents:

agent/
data/
evaluation/
policy/
manifest.json
agent/notebook21_export.py
data/deck.csv
evaluation/notebook22_export.py
policy/notebook20_export.py

Archive verification passed.


# Cell 14 — Final validation

In [16]:
validation = {
    "manifest": MANIFEST_FILE.exists(),
    "zip_archive": ZIP_FILE.exists(),
    "agent_export": (PACKAGE_STRUCTURE["agent"] / "notebook21_export.py").exists(),
    "evaluation_export": (PACKAGE_STRUCTURE["evaluation"] / "notebook22_export.py").exists(),
    "policy_export": (PACKAGE_STRUCTURE["policy"] / "notebook20_export.py").exists(),
    "deck": (PACKAGE_STRUCTURE["data"] / "deck.csv").exists(),
}

for name, passed in validation.items():
    print(
        f"{'[OK]' if passed else '[FAIL]'} {name}"
    )

assert all(validation.values())

print("\nNotebook 23 validation passed.")

[OK] manifest
[OK] zip_archive
[OK] agent_export
[OK] evaluation_export
[OK] policy_export
[OK] deck

Notebook 23 validation passed.


# Cell 15 — Final summary

In [17]:
print("=" * 72)
print("Notebook 23 — Submission Packaging")
print("=" * 72)

print("Repository cards:", manifest["repository_cards"])
print("Official cards:", manifest["official_cards"])
print("Deck size:", manifest["deck_size"])

print()

print("Files packaged:", len(manifest["files"]))
print("ZIP archive:", ZIP_FILE.name)
print("Manifest:", MANIFEST_FILE.name)

print()

print("Submission folder:", PACKAGE_DIR)
print("ZIP location:", ZIP_FILE)

print()

print("NOTEBOOK 23 COMPLETED SUCCESSFULLY")
print("Ready for Notebook 24.")

Notebook 23 — Submission Packaging
Repository cards: 1267
Official cards: 1267
Deck size: 60

Files packaged: 4
ZIP archive: team_jesus_submission.zip
Manifest: manifest.json

Submission folder: D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge\submission_work\team_jesus_notebook23\package
ZIP location: D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge\submission_work\team_jesus_submission.zip

NOTEBOOK 23 COMPLETED SUCCESSFULLY
Ready for Notebook 24.
